# Metadata

> Metadata introspection for the LavaSR plugin used by cjm-ctl to generate the registration manifest.

In [ ]:
#| default_exp meta

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import sys
from typing import Any, Dict
from cjm_media_plugin_lavasr import __version__

In [ ]:
#| export
def get_plugin_metadata() -> Dict[str, Any]:  # Plugin metadata for manifest generation
    """Return metadata required to register this plugin with the PluginManager."""
    # Fallback base path (current behavior for backward compatibility)
    base_path = os.path.dirname(os.path.dirname(sys.executable))
    
    # Use CJM config if available, else fallback to env-relative paths
    cjm_data_dir = os.environ.get("CJM_DATA_DIR")
    
    # Plugin data directory
    plugin_name = "cjm-media-plugin-lavasr"
    if cjm_data_dir:
        data_dir = os.path.join(cjm_data_dir, plugin_name)
    else:
        data_dir = os.path.join(base_path, "data")
    
    db_path = os.path.join(data_dir, "lavasr_jobs.db")
    
    # Ensure data directory exists
    os.makedirs(data_dir, exist_ok=True)
    
    return {
        "name": plugin_name,
        "version": __version__,
        # T24: non-empty description required by the substrate validator (SG-6 / V1 gate).
        "description": "LavaSR v2 speech enhancement — bandwidth extension + optional denoising to improve speech audio quality before transcription.",
        "type": "media-processing",
        "category": "media",
        "interface": "cjm_media_plugin_system.processing_interface.MediaProcessingPlugin",
        
        "module": "cjm_media_plugin_lavasr.plugin",
        "class": "LavaSRProcessingPlugin",
        
        # Critical: The absolute path to THIS environment's python
        "python_path": sys.executable,
        
        "db_path": db_path,
        
        # Phase 5a / CR-7 reframe: binary hard-facts only (quantitative amounts dropped,
        # V12 gate). requires_gpu is False — LavaSR has a real CPU path (device='auto'
        # falls back to CPU when CUDA is absent); it is GPU-optional, not GPU-required.
        # This also resolves the pre-overhaul contradiction (requires_gpu:False +
        # min_gpu_vram_mb:512).
        "resources": {
            "requires_gpu": False
        },
        
        # Track 19: CUDA_VISIBLE_DEVICES (static) + HF_HOME (templated) are declared on the
        # plugin class via WORKER_ENV; the substrate resolves + injects them at spawn.
        "env_vars": {}
    }

## Testing

In [ ]:
import json

metadata = get_plugin_metadata()
print(json.dumps(metadata, indent=2))

{
  "name": "cjm-media-plugin-lavasr",
  "version": "0.0.1",
  "type": "media-processing",
  "category": "media",
  "interface": "cjm_media_plugin_system.processing_interface.MediaProcessingPlugin",
  "module": "cjm_media_plugin_lavasr.plugin",
  "class": "LavaSRProcessingPlugin",
  "python_path": "/home/innom-dt/miniforge3/envs/cjm-media-plugin-lavasr/bin/python3.12",
  "db_path": "/home/innom-dt/miniforge3/envs/cjm-media-plugin-lavasr/data/lavasr_jobs.db",
  "resources": {
    "requires_gpu": false,
    "min_gpu_vram_mb": 512,
    "recommended_gpu_vram_mb": 1024,
    "min_system_ram_mb": 2048
  },
  "env_vars": {
    "HF_HOME": "/home/innom-dt/miniforge3/envs/cjm-media-plugin-lavasr/.cache/huggingface"
  }
}


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()